In [1]:
!pip install transformers torch tqdm


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install --upgrade transformers


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [15]:
OUT_DIR = "../../outputs/final/"
INT_DIR = "../../outputs/intermediate/"

In [16]:
df = pd.read_csv(INT_DIR + "openalex_metadata_full.csv")

print("Total rows:", len(df))
print("Null abstracts:", df['abstract'].isna().sum())

Total rows: 4860
Null abstracts: 618


In [10]:
df = df.dropna(subset=["abstract"])
df = df[df["abstract"].str.strip() != ""]

print("Remaining rows after cleaning:", len(df))

Remaining rows after cleaning: 4242


In [11]:
paper_ids = df["global_paper_id"].tolist()
abstracts = df["abstract"].tolist()

In [12]:
## Load SciBERT
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 605.98it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical ar

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31090, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [13]:
## Embedding Function
## SciBERT outputs token embeddings.
## We take mean pooled representation.
def get_embeddings(text_list, batch_size=16, max_length=512):
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size)):
            batch = text_list[i:i+batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)

            # last hidden state
            last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

            # mean pooling
            attention_mask = inputs["attention_mask"]
            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

            summed = torch.sum(last_hidden * mask_expanded, dim=1)
            summed_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

            mean_pooled = summed / summed_mask

            embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings)

In [14]:
## Generate Embeddings
abstract_embeddings = get_embeddings(abstracts, batch_size=16)

print("Embedding shape:", abstract_embeddings.shape)

100%|██████████| 266/266 [13:52<00:00,  3.13s/it]

Embedding shape: (4242, 768)


In [20]:
## Save
np.save(INT_DIR + "paper_ids.npy", np.array(paper_ids))
np.save(INT_DIR + "abstract_embeddings.npy", abstract_embeddings)

print("Saved paper_ids.npy and abstract_embeddings.npy")

Saved paper_ids.npy and abstract_embeddings.npy


In [21]:
import numpy as np
np.load(INT_DIR + "paper_ids.npy")

array(['NOVEL_DIA_0', 'NOVEL_DIA_1', 'NOVEL_DIA_2', ...,
       'SKG_SUM_W4291111820', 'NOVEL_SUM_W4285744692',
       'NOVEL_SUM_W4384937545'], shape=(4242,), dtype='<U21')

In [22]:
np.load(INT_DIR + 'abstract_embeddings.npy')

array([[ 0.05511688,  0.00364338,  0.0715813 , ..., -0.2560189 ,
         0.07805595, -0.58019584],
       [ 0.05394219, -0.03676981, -0.00789885, ...,  0.0241911 ,
         0.06720224, -0.6158218 ],
       [-0.51626706,  0.19956115, -0.05647893, ...,  0.25275993,
        -0.23334508, -0.639319  ],
       ...,
       [ 0.19874835, -0.03274561, -0.23155253, ..., -0.2483547 ,
        -0.42340037, -0.74607664],
       [-0.06312133, -0.28369245, -0.6304968 , ..., -0.23646729,
         0.0854361 , -0.850594  ],
       [ 0.23740667, -0.3614845 , -0.23021358, ..., -0.38969702,
        -0.17306694, -0.8018435 ]], shape=(4242, 768), dtype=float32)

In [23]:
## Diagonal should be ~1.
## Others < 1.
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(np.load(INT_DIR + 'abstract_embeddings.npy')[:5])
print(sim_matrix)

[[1.         0.9053476  0.7392998  0.88580227 0.90820146]
 [0.9053476  0.9999999  0.78287077 0.92656475 0.86777604]
 [0.7392998  0.78287077 1.0000001  0.7357355  0.71477026]
 [0.88580227 0.92656475 0.7357355  1.         0.8532336 ]
 [0.90820146 0.86777604 0.71477026 0.8532336  1.0000002 ]]
